<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment1/DNNAssignment1_mobilenetv2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

In [ ]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
input_shape = (32, 32, 3)
mobilenetv2_model = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        mobilenetv2_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 127)            │        65,151 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 127)            │           508 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 127)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         2,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,984,123 (11.38 MB)

 Trainable params: 2,948,733 (11.25 MB)

 Non-trainable params: 35,390 (138.24 KB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 30
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('mobilenetv2_model.keras')

Epoch 1/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 83s 104ms/step - accuracy: 0.7322 - loss: 0.8827 - val_accuracy: 0.4700 - val_loss: 2.1425
Epoch 2/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.7439 - loss: 0.8180 - val_accuracy: 0.4260 - val_loss: 2.6730
Epoch 3/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.7669 - loss: 0.7542 - val_accuracy: 0.5230 - val_loss: 2.0630
Epoch 4/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.7855 - loss: 0.6759 - val_accuracy: 0.5750 - val_loss: 1.7730
Epoch 5/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8114 - loss: 0.6043 - val_accuracy: 0.5190 - val_loss: 2.2072
Epoch 6/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8189 - loss: 0.5938 - val_accuracy: 0.5250 - val_loss: 2.0322
Epoch 7/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.8282 - loss: 0.5597 - val_accuracy: 0.4470 - val_loss: 2.4785
Epoch 8/30
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8415 - loss: 0.4893 - val_a

In [ ]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.5446 - loss: 1.5710


[1.632869005203247, 0.5339999794960022]